Persistence means ability to save and restore the state of workflow.

it also stores the intermediate values not just final values
fault tolerance: I can start from where the workflow has last crashed

we can save it in database

Checkpointer: Checkpoints save the values of state at each step
 




In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.9,
    max_tokens=220
)

In [3]:
class JokeState(TypedDict):
    joke:str
    topic: str
    explaination: str

In [4]:
def generate_joke(state: JokeState):
    prompt = f"write a joke for the given topic :- {state['topic']}"
    response = llm.invoke(prompt).content

    return {'joke': response}

In [5]:
def generate_explaination(state: JokeState):
    prompt = f"write an explanation for the joke - {state['joke']}"

    response = llm.invoke(prompt).content

    return {'explaination': response}

In [6]:
graph = StateGraph(JokeState)
graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explaination',generate_explaination)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke','generate_explaination')
graph.add_edge('generate_explaination',END)

checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)




In [7]:
config1 = {"configurable" : {"thread_id": "1"}}
initial_state = {
    "topic": "pizza",
    "joke": "",
    "explaination": ""
}

workflow.invoke(initial_state,config=config1)


{'joke': 'Why did the pizza go to therapy?  \n\nBecause it kept feeling *crust*y and wanted a *cheesy* conversation! 🍕😄',
 'topic': 'pizza',
 'explaination': '**Explanation of the joke**\n\n- **Setup:** “Why did the pizza go to therapy?”  \n  The set‑up invites a question that sounds like a normal “why” story, but the answer will hinge on wordplay rather than a literal reason.\n\n- **Punchline:** “Because it kept feeling *crust*y and wanted a *cheesy* conversation!”  \n  The humor comes from two simultaneous puns:\n\n  1. **“Crusty”** – In everyday English, *crusty* can mean “grumpy, irritable, or in a bad mood.”  A pizza’s crust is literally its outer edge, so the joke pretends the pizza is literally “feeling crusty”'}

In [8]:
workflow.get_state(config1)

StateSnapshot(values={'joke': 'Why did the pizza go to therapy?  \n\nBecause it kept feeling *crust*y and wanted a *cheesy* conversation! 🍕😄', 'topic': 'pizza', 'explaination': '**Explanation of the joke**\n\n- **Setup:** “Why did the pizza go to therapy?”  \n  The set‑up invites a question that sounds like a normal “why” story, but the answer will hinge on wordplay rather than a literal reason.\n\n- **Punchline:** “Because it kept feeling *crust*y and wanted a *cheesy* conversation!”  \n  The humor comes from two simultaneous puns:\n\n  1. **“Crusty”** – In everyday English, *crusty* can mean “grumpy, irritable, or in a bad mood.”  A pizza’s crust is literally its outer edge, so the joke pretends the pizza is literally “feeling crusty”'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1bc29c-2853-6c75-8002-505c5cfe3e0b'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-29T17:18:16.838872+00:00', parent_config={'

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'joke': 'Why did the pizza go to therapy?  \n\nBecause it kept feeling *crust*y and wanted a *cheesy* conversation! 🍕😄', 'topic': 'pizza', 'explaination': '**Explanation of the joke**\n\n- **Setup:** “Why did the pizza go to therapy?”  \n  The set‑up invites a question that sounds like a normal “why” story, but the answer will hinge on wordplay rather than a literal reason.\n\n- **Punchline:** “Because it kept feeling *crust*y and wanted a *cheesy* conversation!”  \n  The humor comes from two simultaneous puns:\n\n  1. **“Crusty”** – In everyday English, *crusty* can mean “grumpy, irritable, or in a bad mood.”  A pizza’s crust is literally its outer edge, so the joke pretends the pizza is literally “feeling crusty”'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1bc29c-2853-6c75-8002-505c5cfe3e0b'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-29T17:18:16.838872+00:00', parent_config={

In [11]:
config2 = {"configurable" : {"thread_id": "2"}}
workflow.invoke({'topic': "Bus"},config=config2)


{'joke': 'Why did the bus break up with the subway?\n\nBecause it found the subway too *concrete*—it wanted something that *actually* *moves* its feelings! 🚍😄',
 'topic': 'Bus',
 'explaination': '**What’s going on in the joke?**\n\n| Element | How it works | Why it’s funny |\n|---------|--------------|----------------|\n| **The “relationship”** | The bus and the subway are personified as if they’re dating. | Turning inanimate transportation into lovers is already absurd, which sets the stage for a pun. |\n| **The word “concrete”** | 1. *Concrete* is literally the material the subway tracks are made of. 2. Figuratively, “concrete” means “solid, unchanging, or literal.” | The double meaning lets the punchline play on both the subway’s physical reality and its emotional flatness. |\n| **The phrase “actually moves its feelings”** | 1. A bus *moves* physically (it goes from stop'}

In [12]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'joke': 'Why did the bus break up with the subway?\n\nBecause it found the subway too *concrete*—it wanted something that *actually* *moves* its feelings! 🚍😄', 'topic': 'Bus', 'explaination': '**What’s going on in the joke?**\n\n| Element | How it works | Why it’s funny |\n|---------|--------------|----------------|\n| **The “relationship”** | The bus and the subway are personified as if they’re dating. | Turning inanimate transportation into lovers is already absurd, which sets the stage for a pun. |\n| **The word “concrete”** | 1. *Concrete* is literally the material the subway tracks are made of. 2. Figuratively, “concrete” means “solid, unchanging, or literal.” | The double meaning lets the punchline play on both the subway’s physical reality and its emotional flatness. |\n| **The phrase “actually moves its feelings”** | 1. A bus *moves* physically (it goes from stop'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': 

Short Term Memory \n
\n Human In the Loop
\n Fault Tolerance
\n Time Travel